In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
train = pd.read_csv("/content/sample_data/mnist_train_small.csv", header=None)
test = pd.read_csv("/content/sample_data/mnist_test.csv", header=None)
lr = 0.001

In [ ]:
# train.iloc[:,1:] /= 255.0

In [ ]:
def get_at(idx):
  return train.iloc[idx, 1:]
def get_label_at(idx):
  return train.iloc[idx, 0]

def y_at(idx):
  y = np.zeros((10,1))
  y[get_label_at(idx)] = 1
  return y
def display(idx):
  plt.imshow(np.array(get_at(idx)).reshape(28, 28), cmap="gray")
  plt.show()

In [ ]:
print(set(get_at(0)))
print(get_at(0).shape)
display(0)

In [ ]:
def relu(Z):
  return np.maximum(0, Z)

def relu_derivative(Z):
  return Z > 0

def softmax(z):
    z = z - np.max(z, axis=0, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=0, keepdims=True)

In [ ]:
class Layer:
  def __init__(self,ip,op,prev_layer=None):
    self.X = None
    self.W = np.random.randn(op, ip) * np.sqrt(2.0 / ip)
    self.b = np.zeros((op,1))
    self.Z = None
    self.A = None
    self.prev_layer = prev_layer

  def forward(self, X):
    self.X = X.reshape(-1, 1)
    self.Z = self.W @ self.X + self.b
    # print("Z shape:", self.Z.shape)
    self.A = relu(self.Z)
    return self.A

  def backward(self,delta):
    if self.prev_layer is None:
      return
    dW = delta @ self.X.T
    db = delta
    self.W -= lr * dW
    self.b -= lr * db
    delta = (self.W.T @ delta) * relu_derivative(self.prev_layer.Z)
    return self.prev_layer.backward(delta)

In [ ]:
l1 = Layer(784, 128)
l2 = Layer(128, 10,prev_layer=l1)

def forward(input):
  l1_output = l1.forward(input)
  l2_output = l2.forward(l1_output)
  return softmax(l2_output)

In [ ]:
loss = 0

In [ ]:
for i in range(len(train)):
  X = np.array(get_at(i))
  Y = y_at(i)

  out = forward(X)
  loss += -np.sum(Y * np.log(out + 1e-12))
  delta = out - Y
  l2.backward(delta)

In [ ]:
loss/=1000
print("Loss:" , loss)

In [ ]:
i = 1

test_y = y_at(i)
text_x = np.array(get_at(i))
out = forward(text_x)
print(np.argmax(out), np.argmax(test_y))
display(i)